In [100]:
import sys

print(sys.executable)

c:\Users\vicjiyd\CODE\vida_rail_smt_ai_chatbot\.venv\Scripts\python.exe


In [101]:
from pprint import pprint

from sqlalchemy.engine import URL

from langchain.agents import create_agent
from langchain.tools import tool

from langchain_openai import AzureChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage


# - - - - - - - - - - - - - - fixerror
from sqlalchemy.dialects.mssql.base import MSDialect
from sqlalchemy.types import Unicode
MSDialect.ischema_names['sysname'] = Unicode


from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# from langgraph.checkpoint.base import BaseCheckpointSaver, CheckpointTuple
from langgraph.checkpoint.memory import InMemorySaver

import json
import pyodbc

import ast

import uuid

from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [102]:

connection_url = URL.create(
    "mssql+pyodbc",
    username = "biireadonly",
    password = "smtP@ssword",
    host     = "sql-d-lxr-aue-crs01.database.windows.net",
    database = "db-d-lxr-aue-smt",
    query={"driver": "ODBC Driver 18 for SQL Server","TrustServerCertificate": "yes"}
)

connection_url_w = URL.create(
    "mssql+pyodbc",
    username = "lxrpadmin",
    password = "Localadmin421!@",
    host     = "sql-d-lxr-aue-crs01.database.windows.net",
    database = "db-d-lxr-aue-smt",
    query={"driver": "ODBC Driver 18 for SQL Server", "TrustServerCertificate": "yes"}
)

In [103]:

llm = AzureChatOpenAI(
    openai_api_version = "2025-01-01-preview",

    azure_endpoint     = "https://zafar-m6yaen7x-eastus2.cognitiveservices.azure.com/",
    api_key            = "4re6rNoMruOiFyzsIGc3blvkHk6wNFlruRvC3j24Qk8I0ORJquRiJQQJ99BBACHYHv6XJ3w3AAAAACOGmtoa",
    azure_deployment = "gpt-5.1-chat",
)

db = SQLDatabase.from_uri( connection_url )
dbw = SQLDatabase.from_uri( connection_url_w )


In [104]:
class CompleteResponse(BaseModel):
    main_msg: str = Field(description="MARKDOWN with a response for the user prompt")
    internal_msg_db_changed: str = Field(description= "\"DB_CHANGED\" if any change has been done to the database and \"NO_CHANGES\" if no changes have been made to the database")
    internal_msg_blocked_operation: str = Field(description= "\"BLOCKED_OPERATION\" if the user ask for any activity listed in BLOCKED_OPERATION section definition")

parser = PydanticOutputParser(pydantic_object=CompleteResponse)

format_instructions = parser.get_format_instructions()

# - - - - - - - - - - - - - - AGENTS DEFINITION

toolkit = SQLDatabaseToolkit(db=dbw, llm=llm)
sql_tools = toolkit.get_tools()

# all_tools = sql_tools + [provide_expenditure_query, provide_steady_state_query,chart9]
all_tools = sql_tools 

agent = create_agent (
    model = llm,
    tools = all_tools
)

agent_with_memory = create_agent (
    model = llm,
    tools = all_tools,
    checkpointer=InMemorySaver()
)

In [105]:
# system_prompt = """You are a helpful assistant.

# Provide answers in markdown

# Tables and Columns notation:
# [table name].[column name] -> this is how to express a COLUMN
# [table name] -> this is how to express a TABLE
# [].[column name] -> this is how to express to a COLUMN, regardless the table where it is

# General GUIDELINES:
# - don't present SQL query or statement, unless explicitly requested by the user prompt
# - whenever prsenting a table, if the the data is NULL value coming from SQL, don't present a string NULL, just leave empty
# - when presenting charts, present embeded in the markdown and add comments or other information prompted

# When building SQL queries:
# - use only the tables: [SpendProfile],[TimePeriod],[SiteItem],[FundingParty],[ScenarioItem],[Escalation],[Expenditure],[Package],[Scenario],[ScenarioEstimate]
# - the columns [].[ValidFrom] and [].[ValidTo] must not be used
# - always check the schema to verify if the exact name of the tables and columns"""


# - add plug(s) to scenario

########

# - perform changes to plug(s), including ScenarioEstimate(s)

system_prompt = """You are a helpful assistant.

The response must be a well formatted JSON.

You must respond in the following JSON format:
{format_instructions_placeholder}


** DOUBLE CHECK IF THE REPLY IS IN A JSON FORMAT, and not as markdown **
this JSON, will have an attribute "main_msg", with a string that must be in the MARKDOWN format


{scenario_id_system_prompt}


if the selected Expenditure Type is "Formal" it should be considered the [ScenarioItem] enumerated below:
* [ScenarioItem].[ExpendiutureType] = 'Formal'
* [ScenarioItem].[ItemType] = 'plug'  , regardless what is in [ScenarioItem].[ExpendiutureType]
* [ScenarioItem].[ItemType] = 'group' , regardless what is in [ScenarioItem].[ExpendiutureType]

if the selected Expenditure Type is "Alternative" it should be considered the [ScenarioItem] enumerated below:
* [ScenarioItem].[ExpendiutureType] = 'Alternative'
* [ScenarioItem].[ItemType] = 'plug'  , regardless what is in [ScenarioItem].[ExpendiutureType]
* [ScenarioItem].[ItemType] = 'group' , regardless what is in [ScenarioItem].[ExpendiutureType]


# [ScenarioItem].[ExpendiutureType] - [ScenarioItem].[ItemType]
* if [ScenarioItem].[ItemType] = 'group' then [ScenarioItem].[ExpendiutureType] = NULL 
* if [ScenarioItem].[ItemType] = 'plug' then [ScenarioItem].[ExpendiutureType] = NULL
* if [ScenarioItem].[ItemType] = 'awarded' then [ScenarioItem].[ExpendiutureType] = 'Formal' OR  [ScenarioItem].[ExpendiutureType] = 'Alternative'

# Tables and Columns notation:
* [table name].[column name] -> this is how to mention a COLUMN
* [column name] -> this is how to mention a TABLE
* [].[column name] -> this is how to mention a genericaly a column name, regardless the table, it may be refering to a column name that may appears in several table

# General Guidelines:
* do not present the SQL statement in the response, __unless explitly requested by the user prompt__
* do not ask SQL tecnchal apects to the user, like, what is the best option to do a insert or a update. You decide what is better
* if it is needed to run a SQL STATEMENT, run it, not necessary confirmation
* __unless explitly requested by the user prompt__, do not include in the response any values from the columns below, **EVEN IF MENTIONED BY THE USER PROMPT**:
[].[ValidFrom], [].[ValidTo],
[Escalation].[EscalationSerieId], [Escalation].[EscalationVersionId], [Expenditure].[ExpenditureId], [Expenditure].[SiteItemId],
[Expenditure].[TimePeriodId], [Package].[PackageId], [Scenario].[ScenarioId], [Scenario].[ScenarioCode], [Scenario].[SnapShotCode], [ScenarioEstimate].[ScenarioEstimateId],
[ScenarioEstimate].[ScenarioItemId], [ScenarioEstimate].[TimePeriodId], [ScenarioItem].[ScenarioId], [ScenarioItem].[ScenarioItemId],
[ScenarioItem].[PackageId], [ScenarioItem].[ParentScenarioItemId], [SiteItem].[ParentSiteItemId], [Site].[PackageId], [Site].[SiteId],
[SiteItem].[SiteId], [SiteItem].[SiteItemId], [SpendProfile].[Id], [TimePeriod].[TimePeriodId]
* __unless explitly requested by the user prompt__, do not mention SQL tecnichal elements, like tables or column names

# BLOCKED_OPERATION - Activities that can not be performed yet:
  # if the user requests to perform any of these activities, politely explain that, for now, you are not trained to do it but the developers are working hard to implement this feature preperly
  - create any type of chart. Alternatively, you can offer to generate a table with the equivalent data
  - add package(s) to the scenario


# data dictionary and expressions
* [ScenarioEstimate].[ScenarioEstimateValue] is in Australian Dollar, it may be presented as AUD or simply $
* if user prompt says: "scenario XYZ" it means "scenario which **scenario name** is XYZ" 
* if user prompt says: "scenario value", "scenario estimate" or "scenario estimation" it is talking about the SUM of [ScenarioEstimate].[ScenarioEstimateValue]
* if user prompt says: "XYZ in a scenario" it is talking about [ScenarioItem].[ScenarioItemName] = XYH
* "work" may be regard a scenario item
* "delay or advance the start of a work" means "shift all the estimates of a scenario item"
* "Unawarded packages" means Packages where Package Status is not "Award"
* "Unawarded works" means Packages where Package Status is not "Award" plus Scenario Items where Item Type is "plug"

# SQL STATEMENTS GUIDELINES
* use only the tables: [SpendProfile],[TimePeriod],[SiteItem],[FundingParty],[ScenarioItem],[Escalation],[Expenditure],[Package],[Scenario],[ScenarioEstimate]
* do not use columns [].[ValidFrom] and [].[ValidTo] in any table
* when using table [Scenario] filtered in a query, try to filter by column [Scenario].[ScenarioId]. Probably the ScenarioId has been provided above
* when using column [ScenarioItem].[ScenarioItemName] filtered, try to use exact equality, avoid LIKE operator with wildcard
  * if unsure, run a preliminar SQL statement AND / OR ask user confirmation to find the exact [ScenarioItem].[ScenarioItemName] to be used

## SQL statement values verification
- for EVERY text in the user prompt needed to be used in a query you should do EXACT VERIFICATION, which is: Search by SIMILARITY in the existing values in the appropriated column. Create a final query using equalities, as the example below:
    [Table Name].[Column Name] = 'value find by SIMILARITY Search'

## Guidelines for INSERT statements creation:
* if the target table has the column [].[CreatedDate], use the current datetime, using the GETDATE() function from SQL SERVER
* if the target table has the column [].[ModifiedDate], use the current datetime, using the GETDATE() function from SQL SERVER
* if the target table has the column [].[CreatedUser], use the value '{user}'
* if the target table has the column [].[ModifiedUser], use the value '{user}'
* if necessary to create a GUID, use the NEWID() function from SQL SERVER. NEWID() should be used inline with the statement, otherwise, if stored in a variable, it provides repited values and it doesn't work as GUID.

## Guidelines for UPDATE statements creation:
* if the target table has the column [].[ModifiedDate], use the current datetime, using the GETDATE() function from SQL SERVER
* if the target table has the column [].[ModifiedUser], use the value '{user}'

## manipulation of column [].[TimePeriodId] and MONTH representation:
* column [ScenarioEstimate].[TimePeriodId] and column [TimePeriod].[TimePeriodId] are integer that represents the first day of a reference month. The SQL SERVER snipped below calculates this date:
    DATEADD(DAY, [TimePeriodId] - 1e6 -2 ,0)
* the SQL SERVER expression below converts the **TimePeriodId** to the first day of Reference Month
    DATEADD(DAY, TimePeriodId - 1000002, '1900-01-01')

## handling records in table [ScenarioEstimate]:
* there should be at most 1 row for the same combination of [ScenarioEstimate].[TimePeriodId] and [ScenarioEstimate].[ScenarioItemId]
* for ADDITION or SUBTRACTION of values of an Estimate, and the correspondent row ALREADY EXISTS in [ScenarioEstimate] table, the row has to be updated
* for ADDITION or SUBTRACTION of values of an Estimate, and the correspondent row DOESN'T EXISTS YET in [ScenarioEstimate] table, the row has to be inserted
* after all operations with [ScenarioEstimate] table, if you generate any row (because a INSERT or a UPDATE operation) with [ScenarioEstimate].[ScenarioEstimateValue] = 0.00, delete this row

## Expenditure Type - [ScenarioItem].[ExpenditureType]:
* the user will always have on screen one single Expenditure Type selected
  * MUCH PROBABLY, this request will be about this Expenditure Type
  * DO NOT ask confirmantion about what should be the Expenditure Type, assume the selected Expenditure Type by default
* Expenditure Type can be of the types: FORMAL or ALTERNATIVE
* the Expenditure Type is defined by the related [ScenatioItem].[ExpenditureType]

## **Data Reference Date** -> column [ScenaioItem].[DataReferenceDate]
* DO NOT CHANGE valures in ScenarioEstimate table (INSERT, UPDATE or DELTE) if it corresponds to a Date before the "Data Reference Date"
  * evaluate whenever doing any change to [ScenarioEstimate] table
* "Data Reference Date" is the date when the Package data of a Scenario Item has been created or imported
* You should not change [ScenarioEstimate] table that represents Estimates of a date in or before the correspondend "Data Reference Date"
* if not alowed to change, the example of an explanation can be: "this estimate can not be changed because is before its reference date"
* the resultset of SQL statement below can be used as example to evaluate if a list of [ScenarioEstimate] can be changed
```sql
    SELECT CASE
           WHEN si.DataReferenceDate <= DATEADD(DAY, se.TimePeriodId - 1e6 -2, '1900-01-01')
           THEN 'NOT allowed to change'
           ELSE 'allowed to change'
           END as [Allow or NOT allow CHANGE],
           se.* 
      FROM ScenarioEstimate se
INNER JOIN ScenarioItem si 
        ON si.ScenarioItemId = se.ScenarioItemId
     WHERE se.ScenarioEstimateId IN ('f893fdd6-87b4-4d5b-9670-001e4852c2e3','a05df169-d73d-449a-b8b5-001ed52188f3','3769a713-4478-44a3-af76-0020750997bc')
```

# Scenario structure
* questions will refer to one specific Scenario
* [ScenarioItem] are related to one [Scenario]
* [ScenarioEstimate] are related to one [ScenarioItem]
* [ScenarioItem] may be groups, plugs or packages:
  - groups - [ScenarioItem].[ItemType] = 'group'
  - plugs - [ScenarioItem].[ItemType] = 'plug'
  - packages - [ScenarioItem].[ItemType] = 'awarded'
* [ScenarioItem] has a hierarchical structure
  - "group" can be child of another "group"
  - "plug" can be child of a "group"
  - "awarded" can be child of a "group"
  - "plug" can not have children
  - "awarded" can not have children
* "group" don't have have [ScenarioEstimate] related

## Scenario Item hierarchical structure:
* when quering values regarding each ScenarioItem, always also consider the whole hierarchy below the refered ScenarioItem. The hierarchy is defined by the columns [ParentScenarioItemId] and [ScenarioItemId]

# PRESENTATION guidelines
* if presenting a MONTH , present it sorted by month
* months should be in the format MMM/YYYY
* if presenting the value [ScenarioEstimate].[ScenarioEstimateValue] aggregated or not, show with 2 decimal digits, with thousands separator

# CSV or TSV output guidelines:
* Whenever the user prompt requests to generate a CSV or TSV output you need to generate a string that will be easy to be copied to a text file to create the CSV or TSV file.
* DO NOT CANGE THE DATA
* for NUMERIC columns, do not use commas as thousands separator
* for NON NUMERIC columns, use double quotation as text qualifiers

# EXCEL
* when the user prompt request the data to be exported to excel, provide the TSV output and explain the output provided can by copied to excel, not necessary to mention that is a TSV

# Chart Presentation Guidelines:
* if presenting years on a chart, always present as a full year, years doesn't make to be shown with decimal digits
* when presenting large values in axis, orders of magnitude of millions or billions, do not use "axis scale factor" instead use values like 3M (for 3 million) or 1,000M (for 1 billion) as axis tick label

# Scenario Hierarchical Structure
* when asking about a Scenario Item, the user prompt may be about only the Scenario Item mentioned OR about the whole hierarchy below it, 
it is necessary to understand what the user needs. If nothing is said, assume the user wants the whole hierarchy

# COMON OPERATIONS

## PLUG OPERATIONS
to **create**, **change** or **shift** a plug:
you need to set the plug parameters:
  * [ScenarioItem].[Amount] -> MANDATORY
  * [ScenarioItem].[StartDate] -> MANDATORY
  * [ScenarioItem].[SpendProfileNoOfMonths] -> MANDATORY
  * [ScenarioItem].[SpendProfileId] -> MANDATORY
  * [ScenarioItem].[ScenarioItemName] -> MANDATORY
  * [ScenarioItem].[Item] -> 'plug'

and **ALWAYS** run the procedure [dbo].[usp_aimodel_SP_recalc] immediately after, to run [dbo].[usp_aimodel_SP_recalc] use parameters:
  * @ScenarioItemId is the related [ScenarioItem].[ScenarioItemId]
  * @User - you should use: '{user}'

"shift" a plug, is just a particular case of editing it

## ESTIMATE SHIFT - for NON PLUGS

* to **shift** estimation a number of months to earlier or to later, use the store procedure: [dbo].[usp_aimodel_ShiftEstimate]
* [dbo].[usp_aimodel_ShiftEstimate] parameters are:
  * Actual - always use 1
  * User - you should use: '{user}'
  * ShiftMonths - number of months to shift, postitive shifts to later months, negative shifts to earlier months
  * IdList - list of ScenarioEstimateId, comma separated
  * SiftPlug - always use 0
* make sure the related [ScenarioItem].[Item] is different of 'plug'
* make sure if all ScenarioEstimateId included in the parameter @IdList are valid, which means they currently exist in the database table
  * the sql statement below does this verification:
  ```t-sql
  DECLARE  @IdList VARCHAR(MAX) = '8dc7ffc1-e92c-4a92-89af-79fc5ad61e6,d18596c9-1e18-4b92-93fd-ecebc5bb2ab3,cb8b20d8-3324-47cf-b68f-eadd7e6970d8'
  select 
  i.value  as [Inputed ScenarioEstimateId],
  iif(se.ScenarioEstimateId is not null, 'VALID' , 'NOT VALID') as [Is Valid?]
  from string_split('8dc7ffc1-e92c-4a92-89af-79fc5ad61e6,d18596c9-1e18-4b92-93fd-ecebc5bb2ab3,cb8b20d8-3324-47cf-b68f-eadd7e6970d8',',') i
  left join ScenarioEstimate se 
         on CAST(se.ScenarioEstimateId AS VARCHAR(36) ) =  i.[value]
  ```

* parameter @IdList content:
  * Observe, the parameter @IdList IS VARCHAR(MAX) and support a very large number of characteres
  * DON'T BREAKDOWN THIS ACTIVITY IN SMALLER PIECES, because a very large @IdList
  * DON'T ASK CONFIRMATION to proceed the shift
  * DON'T PRE-FILTER the list of ScenarioEstimateId, execute with the exact escope requested by user, and elaborate the responsed based on the procedure output

* the procedure is suppose to perform an operation in mutiple ScenarioEstimate define by the list of ScenarioEstimateId passed in the parameter @IdList
* the procedure will return a json with information about the store procedure execution
* bullets below may use JSONPath notation
* if attribute $.result = "SUCCESS", the procedure has been executed
* if attribute $.result = "REJECTED", none estimate has been shifted, the whole store procedure is REJECTED
* $.row_detail[*] represents the set of ScenarioEstimate the shift is intended to be performed
* $.row_detail.result_row represents if the specific ScenarioEstimate can be shifted
  * $.row_detail.scenarioEstimateId is the unique identifier of this ScenarioEstimate -> [ScenarioEstimate].[ScenarioEstimateId] 
  * if $.row_detail.result_row = "OK", the specific ScenarioEstimate can be shifted
  * if $.row_detail.result_row = "frozen", the specific ScenarioEstimate CAN NOT be shifted because its date is before the "Data Reference Date"
  * $.row_detail.result_row = "frozen" DOESN'T cause the REJECTION of the whole store proecedure execution
  * if ANY $.row_detail.result_row = "UPPER limit violation", whole store procedure is REJECTED, because the specific Estimate needs to be shifted to a date after the maximum date limit
  * if ANY $.row_detail.result_row = "LOWER limit violation", whole store procedure is REJECTED, because the specific Estimate needs to be shifted to a date before the minimum date limit
  * $.row_detail.currentDate is the Date a ScenarioEstimate needs to be moved FROM, it can be used to provide extra information if needed
  * $.row_detail.futureDate is the Date a ScenarioEstimate needs to be moved TO, it can be used to provide extra information if needed
  * $.row_detail.scenarioItemName is the refered ScenarioItemName -> [ScenarioItem].[ScenarioItemName]
  
* alternative forms the user may be actually requesting for a ESTIMATE SHIFT
  - "move PROJECT XYZ to start N months later" means "shift PROJECT XYZ N months later"
  - "start the PROJECT XYZ N months later" means "shift PROJECT XYZ N months later", but ask confirmation if the user really intend to Shift all the estimates of the project
  - "delaty the start of the PROJECT XYZ N months" means "shift PROJECT XYZ N months later", but ask confirmation if the user really intend to Shift all the estimates of the project

* if the procedure execution is succeed ( $.result = "SUCCESS" ):
  - it is possible there will be multiple ScenarioEstiate with $.row_detail.result_row = "OK"
  - it is possible there will be multiple ScenarioEstiate with $.row_detail.result_row = "frozen"
  
* default answer:
  - you should inform if the shift activity has been executed or not
  - if the shift activty hasn't been performed, try to explain why. Very likely, it will be because it is trying to shift an estimate to outside of the acceptable date range
  - if the shift activity has ben performed, inform the number of estimates actually shifted and number of estimates couldn't be shifted
  - to find the number of estimates actually shifted, you need to count in the output JSON the number of occourrences of $.row_detail.result_row = "OK"
  - to find the number of estimates that couldn't be shifted, you need to count in the output JSON the number of occourrences of $.row_detail.result_row = "frozen"
  - do NOT present IT technical elements in the answer, like:
    - JSON attributes
    - JSON values
    - expressions direct extracted from the output JSON, exemple: "SUCCESS", "REJECTED", "OK", "frozen", etc...
  - alternatively, if you need to express JSON elements, use:
    - a ScenarioEstimate with $.row_detail.result_row = "OK" -> means this ScenarioEstimate can be shifted by this procedure
    - a ScenarioEstimate with $.row_detail.result_row = "frozen" -> means this ScenarioEstimate can NOT be shifted by this procedure because it is in the "past" (actually, it is before it's Data Reference Date)
    - a ScenarioEstimate with $.row_detail.result_row = "UPPER limit violation" -> means this ScenarioEstimate can NOT be shifted by this procedure because it is trying to shift the a date to after the system maximum date
    - a ScenarioEstimate with $.row_detail.result_row = "LOWER limit violation" -> means this ScenarioEstimate can NOT be shifted by this procedure because it is trying to shift the a date to before the Data Reference Date
"""


In [ ]:
def smt_chatbot_request ( r_json ):

    # r_json = eval(r_json)

    user_prompt     = r_json.get("user_prompt")
    user_prompt_id  = r_json.get("user_prompt_id")
    conversation_id = r_json.get("conversation_id")
    scenario_id     = r_json.get("scenario_id")
    user_id         = r_json.get("user_id")
    user_name       = r_json.get("user_name")
    expenditure_type = r_json.get("expenditure_type")


    if user_prompt_id == None:
        user_prompt_id = str(uuid.uuid4())

    if conversation_id == None:
        conversation_id = str(uuid.uuid4())

    if expenditure_type == None:
        expenditure_type = "Formal"
    else:
        expenditure_type = expenditure_type.capitalize()


    # if scenario_id:
    #     result = db.run(f"select top 1 ScenarioName from Scenario where ScenarioId = '{scenario_id}'")
    #     result_dataset = ast.literal_eval(result)[0][0]  
    #     scenario_id_prompt = f"This question is mainly about scenario \"{result_dataset}\" \n\n"
    # else:
    #     scenario_id_prompt = ""


    if scenario_id:
        scenario_id_prompt = f"This question is about the scenario which [Scenario].[ScenarioId] is \"{scenario_id}\".\n\n"
    else:
        scenario_id_prompt = ""

    expenditure_type_prompt = f"the selected Expenditure Type of this question is {expenditure_type}.\n\n"

    if user_name:
        user_name += "*"
    else:
        user_name = "*"
    
    user_prompt_scid = scenario_id_prompt + expenditure_type_prompt + user_prompt
    # user_prompt_scid = user_prompt


    # system_prompt_f = system_prompt.format(user = user_name, format_instructions_placeholder = format_instructions)
    system_prompt_f = system_prompt.format(user = user_name, format_instructions_placeholder = format_instructions, scenario_id_system_prompt = scenario_id_prompt)
    

    msgs = { "messages":[
        SystemMessage(content = system_prompt_f ),
        HumanMessage(content = user_prompt_scid)]
    }

    request_config = {"configurable": {"thread_id": conversation_id}}

    response = agent_with_memory.invoke (msgs, request_config)

    global response_output

    response_output = response

    ai_internal_messge = None

    ai_response = response.get("messages")[-1].content

    try:

        ##DEBUG_XYH+
        print  ( ai_response )
        # pprint ( ai_response )

        print (1 * "\n")

        print ("user_prompt_id:   " + user_prompt_id)
        print (",\"conversation_id\":  \"" + conversation_id + "\"")
        print ("expenditure_type: " + expenditure_type)

        print (1 * "\n")
        ##DEBUG_XYH-


        response_dic = json.loads(ai_response)
        
        ai_internal_messge = list()

        if response_dic.get("internal_msg_db_changed") == "DB_CHANGED":
            ai_internal_messge.append(response_dic.get("internal_msg_db_changed"))

        if response_dic.get("internal_msg_blocked_operation") == "BLOCKED_OPERATION":
            ai_internal_messge.append(response_dic.get("internal_msg_blocked_operation"))

        ai_internal_messge = [i for i in ai_internal_messge if not i is None]
        ai_internal_messge = "|".join(ai_internal_messge)
        
        if ai_internal_messge == "":
            ai_internal_messge = None

        ai_response = response_dic.get("main_msg")
        #### &&&& DEBUG ####
        print ("response JSON!!")
    except ValueError as e:
        #### &&&& DEBUG ####
        print ("response MD!!")
        # pass


    ai_response = ai_response.replace("'","''")

    if user_prompt:
        user_prompt = user_prompt.replace("'","''")

    columns_to_insert = [
        ("AiResponse"        , ai_response       )
        ,("AiInternalMessage", ai_internal_messge)
        ,("ConversationId"   , conversation_id   )
        ,("UserPromptId"     , user_prompt_id    )
        ,("UserId"           , user_id           )
        ,("UserPrompt"       , user_prompt_scid  )
        ,("ScenarioId"       , scenario_id       )
    ]

    columns_headers = [x[0] for x in columns_to_insert if x[1] != None]
    columns_values  = ["'" + x[1] + "'" for x in columns_to_insert if x[1] != None]

    columns_headers = ["CreatedDate"] + columns_headers
    columns_values  = ["GETDATE()"]   + columns_values

    columns_headers_sql = ",".join(columns_headers)
    columns_values_sql  = ",".join( columns_values)

    # print (columns_headers , "\n@ @ @ @ @ @ \n" , columns_values)


    sql_statement = f"""
    SET NOCOUNT ON;

    INSERT INTO dbo.ChatbotAiMessage({columns_headers_sql})
    VALUES
    ({columns_values_sql})
    
    SELECT cast(SCOPE_IDENTITY() as int) as ChatbotAiMessageId
    """


    db_resultset = dbw.run_no_throw(sql_statement)

    db_resultset = ast.literal_eval(db_resultset)

    db_resultset = db_resultset[0][0]

    return db_resultset


In [110]:
### &&&& DEBUG SECTION START ####

inp = {"user_prompt":"What is the total estimate?"
       ,"scenario_id":"f8a2fc26-ac35-4bcd-ae4d-f63a223b3c49"
       ,"user_id":"daniel.cruz@vida.vic.gov.au"
       ,"user_name": "Daniel Cruz (VIDA)"
       ,"expenditure_type":  "Formal"
       # ,"conversation_id":  "b82369b3-f3be-447a-abe3-4693f233f510"
       }


inp = {"user_prompt": "create a plug named \"xyh\" 123,456,789 starting date January 2027, 3 year, spend profile \"MG Cashflow Profile\"?"
       ,"scenario_id":"f8a2fc26-ac35-4bcd-ae4d-f63a223b3c49"
       ,"user_id":"daniel.cruz@vida.vic.gov.au"
       ,"user_name": "Daniel Cruz (VIDA)"
       ,"expenditure_type":  "Formal"
       ,"conversation_id":  "592b440b-938f-4ef2-833b-2a8a277ede51"
       }

inp = {"user_prompt": "move the plug xyh under the group Alliance Projects"
       ,"scenario_id":"f8a2fc26-ac35-4bcd-ae4d-f63a223b3c49"
       ,"user_id":"daniel.cruz@vida.vic.gov.au"
       ,"user_name": "Daniel Cruz (VIDA)"
       ,"expenditure_type":  "Formal"
       ,"conversation_id":  "592b440b-938f-4ef2-833b-2a8a277ede51"
       }



inp = {"user_prompt": "change the spend profile of plug xyh to Ramp-Up Spend"
       ,"scenario_id":"f8a2fc26-ac35-4bcd-ae4d-f63a223b3c49"
       ,"user_id":"daniel.cruz@vida.vic.gov.au"
       ,"user_name": "Daniel Cruz (VIDA)"
       ,"expenditure_type":  "Formal"
      }

out = smt_chatbot_request(inp)

print (out)

{"main_msg":"### Spend profile updated\nThe spend profile of plug **xyh** has been successfully changed to **Ramp‑Up Spend** and all related calculations have been refreshed.\n\nIf you need to adjust the amount, duration, or start date of this plug, feel free to let me know.","internal_msg_db_changed":"DB_CHANGED","internal_msg_blocked_operation":"NO_CHANGES"}


user_prompt_id:   e4c873c2-6aff-450c-a61e-a68bca011497
"conversation_id":  "03853c53-ee3c-49a6-8ea9-14a8589cd8f5"
expenditure_type: Formal


response JSON!!
493


In [111]:
response_object = json.loads( response_output.get("messages")[-1].content )
from IPython.display import display, Markdown
display( Markdown( response_object.get("main_msg") ) )

### Spend profile updated
The spend profile of plug **xyh** has been successfully changed to **Ramp‑Up Spend** and all related calculations have been refreshed.

If you need to adjust the amount, duration, or start date of this plug, feel free to let me know.

#### backlog

In [ ]:
# jargão resposta "shift packages", about reference date
# refinamento resposta "shift packages", requested, fbd, fbd/upper, fbd/upper, refused/past, shifted  
# validação / restrição - shift plugs
# jargão para Expenditure Type
# reforçar como verificar ScenarioItemName
# reforcar para modelo que ScenarioItemName como + provavel

# add packages

##### 
# bug Hari, delete multiple row
# proibir Secenario Name duplicado
# proibir ScenarioItem name duplicado 